RETRAINING FOR PERSONAL BASELINE TO ALWAYS BE UPDATED
this is all code that has been used before 

In [ ]:
from pathlib import Path
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn import preprocessing

class WindowAutoencoder(nn.Module):
    def __init__(self, input_dim=20, latent_dim=2, hidden_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim))
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

def extract_windows(feature_array, window_size=20, step=1):
    windows = []
    for i in range(0, len(feature_array) - window_size, step):
        windows.append(feature_array[i:i + window_size])
    return np.array(windows)

def train_one_autoencoder(data, epochs=100, lr=1e-4):
    tensor_data = torch.tensor(data, dtype=torch.float32)
    dataset = TensorDataset(tensor_data)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
    
    model = WindowAutoencoder()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        for batch in dataloader:
            x = batch[0]
            recon_x, z = model(x)
            loss = F.mse_loss(recon_x, x)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    return model

def retrain_all_models():
    filepaths = list(Path("daria-recordings").glob("*.wav")) + list(Path("daria-recordings/sessions").glob("*.wav"))
    print(f"Retraining on {len(filepaths)} recordings...")
    
    # --- RMS ---
    all_rms = np.vstack([extract_windows(librosa.feature.rms(y=librosa.load(str(fp), sr=44100, mono=True)[0])[0]) for fp in filepaths])
    rms_scaled = preprocessing.MinMaxScaler().fit_transform(all_rms)
    model_rms_new = train_one_autoencoder(rms_scaled)
    torch.save(model_rms_new.state_dict(), "RMS/rms_autoencoder.pt")
    print("RMS model retrained.")

    # --- Centroid ---
    all_centroid = np.vstack([extract_windows(librosa.feature.spectral_centroid(y=librosa.load(str(fp), sr=44100, mono=True)[0], sr=44100)[0]) for fp in filepaths])
    centroid_scaled = preprocessing.MinMaxScaler().fit_transform(all_centroid)
    model_centroid_new = train_one_autoencoder(centroid_scaled)
    torch.save(model_centroid_new.state_dict(), "Centroid/centroid_autoencoder.pt")
    print("Centroid model retrained.")

    # --- Onset ---
    all_onset = np.vstack([extract_windows(librosa.onset.onset_strength(y=librosa.load(str(fp), sr=44100, mono=True)[0], sr=44100)) for fp in filepaths])
    onset_scaled = preprocessing.MinMaxScaler().fit_transform(all_onset)
    model_onset_new = train_one_autoencoder(onset_scaled)
    torch.save(model_onset_new.state_dict(), "ON-set/onset_autoencoder.pt")
    print("Onset model retrained.")

    # --- Baselines (RMS, centroid, onset) ---
    baseline_rms = model_rms_new.encoder(torch.tensor(rms_scaled, dtype=torch.float32)).detach().numpy().mean(axis=0)
    baseline_centroid = model_centroid_new.encoder(torch.tensor(centroid_scaled, dtype=torch.float32)).detach().numpy().mean(axis=0)
    baseline_onset = model_onset_new.encoder(torch.tensor(onset_scaled, dtype=torch.float32)).detach().numpy().mean(axis=0)
    np.save("baseline_rms.npy", baseline_rms)
    np.save("baseline_centroid.npy", baseline_centroid)
    np.save("baseline_onset.npy", baseline_onset)

    # --- Phrase continuity baseline (IOI std) ---
    window = 5
    all_stds = []
    for fp in filepaths:
        y, sr = librosa.load(str(fp), sr=44100, mono=True)
        onset_times = librosa.onset.onset_detect(y=y, sr=sr, units='time')
        if len(onset_times) < window + 1:
            continue
        iois = np.diff(onset_times)
        for i in range(window, len(iois)):
            all_stds.append(np.std(iois[i-window:i]))
    baseline_std_ioi = np.mean(all_stds)
    np.save("baseline_std_ioi.npy", baseline_std_ioi)

    print("All baselines recomputed and saved.")
    print("Retraining complete.")

# retrain_all_models()  # uncomment to actually run